# AethyxLM - Google Colab Training Notebook

Train AethyxLM (GPT-style decoder-only LLM) on TinyStories using Google Colab T4 GPU.

**Model:** 8 layers, 256 dim, 8 heads, 128 context, ~14.5M params
**Dataset:** TinyStories (subset)
**Tokenizer:** BPE 32k vocab

In [ ]:
# Clone repository and install dependencies
!git clone https://github.com/your-username/AethyxLM.git 2>/dev/null || echo 'Repo already cloned or use local upload'
%cd AethyxLM
!pip install tokenizers datasets -q

In [ ]:
# Verify CUDA
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Download and prepare TinyStories dataset
from datasets import load_dataset
import random

print('Loading TinyStories...')
ds = load_dataset('roneneldan/TinyStories', split='train')

# Use subset for faster training (adjust as needed)
NUM_STORIES = 10000  # Increase for full training
texts = ds['text'][:NUM_STORIES]

random.seed(42)
random.shuffle(texts)
split = int(0.9 * len(texts))
train_texts = texts[:split]
val_texts = texts[split:]

with open('data/train.txt', 'w') as f:
    f.write('\n\n'.join(train_texts))
with open('data/val.txt', 'w') as f:
    f.write('\n\n'.join(val_texts))

print(f'Train stories: {len(train_texts)}')
print(f'Val stories: {len(val_texts)}')

In [ ]:
# Train BPE tokenizer (32k vocab)
%cd tokenizer
!python train_tokenizer.py
%cd ..

# Verify tokenizer
from tokenizer.tokenizer import AethyxTokenizer
tok = AethyxTokenizer()
print(f'Vocab size: {tok.vocab_size}')
print(f'Test encode: {tok.encode("Hello world")}')
print(f'Test decode: {tok.decode(tok.encode("Hello world"))}')

In [ ]:
# Update training config for Colab T4
import json

with open('configs/train_config.json') as f:
    cfg = json.load(f)

cfg['training'].update({
    'max_steps': 10000,
    'warmup_steps': 1000,
    'batch_size': 16,
    'grad_accum_steps': 1,
    'use_amp': True,
    'eval_interval': 500,
    'save_interval': 1000,
    'log_interval': 50
})

with open('configs/train_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config updated:')
print(json.dumps(cfg['training'], indent=2))

In [ ]:
# Start training
!python train.py --config configs/train_config.json --device cuda

In [ ]:
# Download best checkpoint
from google.colab import files
files.download('checkpoints/checkpoint_best.pt')
files.download('checkpoints/checkpoint_latest.pt')

In [ ]:
# Quick inference test
import torch
from model.gpt import GPT
from tokenizer.tokenizer import AethyxTokenizer

device = 'cuda'
model = GPT().to(device)
tok = AethyxTokenizer()

# Load best checkpoint
ckpt = torch.load('checkpoints/checkpoint_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        logits = model(ids[:, -128:])
        logits = logits[:, -1, :] / temperature
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        ids = torch.cat([ids, next_id], dim=1)
    return tok.decode(ids[0].tolist())

print(generate('Once upon a time', max_new_tokens=150))